In [1]:
import numpy as np
import torch
import torch.nn.functional as F
from chronos import BaseChronosPipeline
from aeon.transformations.collection.base import BaseCollectionTransformer

2026-03-21 21:38:06.060182: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
dataset = 'Worms'

In [3]:
from sklearn.base import BaseEstimator, TransformerMixin

class MantisEmbedding2(BaseEstimator, TransformerMixin):
    """Extracts MantisV2 frozen embeddings (256d). Resizes input to 512 timesteps."""

    def __init__(self, device="cpu", include_diff=True):
        from mantis.architecture import MantisV2
        from mantis.trainer import MantisTrainer
        self.device = device
        self.include_diff = include_diff
        self.network = MantisV2(device=self.device).from_pretrained("paris-noah/MantisV2")
        self.model = MantisTrainer(device=self.device, network=self.network)

    def fit(self, X, y=None):
        return self

    def _embed(self, X):
        with torch.inference_mode():
            X_r = F.interpolate(
                torch.tensor(X, dtype=torch.float), size=512, mode='linear', align_corners=False
            )
            return self.model.transform(X_r)

    def transform(self, X):
        emb = self._embed(X)
        if self.include_diff:
            diff_emb = self._embed(np.diff(X, axis=-1))
            return np.hstack([emb, diff_emb])
        return emb
    
class Chronos2Embedding2(BaseEstimator, TransformerMixin):
    """Extracts Chronos [REG] token embeddings.

    Chronos-2 (decoder-only): [REG] at [-2]
    Chronos-Bolt (encoder-decoder): [REG] at [-1]
    """
    def __init__(self, model_id="amazon/chronos-2", batch_size=256, device="cpu", include_diff=True):
        self.model_id = model_id
        self.batch_size = batch_size
        self.device = device
        self.include_diff = include_diff
        self._pipeline = None

    @property
    def _is_bolt(self):
        return "bolt" in self.model_id

    def _get_pipeline(self):
        if self._pipeline is None:
            self._pipeline = BaseChronosPipeline.from_pretrained(self.model_id, device_map=self.device)
        return self._pipeline

    def __getstate__(self):
        state = self.__dict__.copy()
        state['_pipeline'] = None
        return state

    def fit(self, X, y=None):
        return self

    def _embed(self, X):
        pipeline = self._get_pipeline()

        with torch.inference_mode():
            reg_idx = -1 if self._is_bolt else -2

            all_embs = []
            for i in range(0, len(X), self.batch_size):
                batch = [torch.from_numpy(x.squeeze(0)).float() for x in X[i:i+self.batch_size]]
                embeddings, _ = pipeline.embed(batch)
                if self._is_bolt:
                    vecs = embeddings[:, reg_idx, :].detach().cpu().float().numpy()
                else:
                    vecs = np.stack([e[0, reg_idx, :].detach().cpu().float().numpy() for e in embeddings])
                all_embs.append(vecs)
            return np.vstack(all_embs)

    def transform(self, X):
        emb = self._embed(X)
        if self.include_diff:
            diff_emb = self._embed(np.diff(X, axis=-1))
            return np.hstack([emb, diff_emb])
        return emb

In [4]:
# Test all embedding transformers
from time import perf_counter
from tscglue import utils, data_loader
from tscglue.models_tsfm import Chronos2Embedding, MantisEmbedding

X_train, y_train, X_test, y_test = data_loader.load_fold(dataset, 4)
print(f"X_test shape: {X_test.shape}\n")

transformers = [
    ("MantisV2", MantisEmbedding()),
    ("MantisV2", MantisEmbedding2()),
    ("MantisV2", MantisEmbedding()),
    ("chronos-bolt-tiny", Chronos2Embedding(model_id="amazon/chronos-bolt-tiny")),
    ("chronos-bolt-tiny", Chronos2Embedding(model_id="amazon/chronos-bolt-tiny")),
    ("chronos-bolt-tiny", Chronos2Embedding2(model_id="amazon/chronos-bolt-tiny")),
    ("chronos-bolt-mini", Chronos2Embedding(model_id="amazon/chronos-bolt-mini")),
    ("chronos-bolt-mini", Chronos2Embedding2(model_id="amazon/chronos-bolt-mini")),
    ("chronos-bolt-small", Chronos2Embedding(model_id="amazon/chronos-bolt-small")),
    ("chronos-bolt-small", Chronos2Embedding2(model_id="amazon/chronos-bolt-small")),
    ("chronos-bolt-base", Chronos2Embedding(model_id="amazon/chronos-bolt-base")),
    ("chronos-bolt-base", Chronos2Embedding2(model_id="amazon/chronos-bolt-base")),
    ("chronos-2", Chronos2Embedding(model_id="amazon/chronos-2")),
    ("chronos-2", Chronos2Embedding2(model_id="amazon/chronos-2")),

]

for name, t in transformers:
    t0 = perf_counter()
    emb = t.transform(X_test)
    elapsed = perf_counter() - t0
    print(f"{name}: {emb.shape} ({elapsed:.1f}s)")

X_test shape: (77, 1, 900)

MantisV2: (77, 512) (1.2s)
MantisV2: (77, 512) (0.4s)
MantisV2: (77, 512) (1.1s)
[Chronos2Embedding.transform] start X.shape=(77, 1, 900)
[Chronos2Embedding] from_pretrained(amazon/chronos-bolt-tiny) took 0.7s
[Chronos2Embedding._embed] batch 0: prep=0.00s embed=0.14s total=0.14s
[Chronos2Embedding._embed] pipeline=0.71s total=0.85s shape=(77, 256)
[Chronos2Embedding.transform] embed done 0.85s
[Chronos2Embedding._embed] batch 0: prep=0.00s embed=0.36s total=0.36s
[Chronos2Embedding._embed] pipeline=0.00s total=0.36s shape=(77, 256)
[Chronos2Embedding.transform] diff_embed done 1.22s
chronos-bolt-tiny: (77, 512) (1.2s)
[Chronos2Embedding.transform] start X.shape=(77, 1, 900)
[Chronos2Embedding] from_pretrained(amazon/chronos-bolt-tiny) took 0.7s
[Chronos2Embedding._embed] batch 0: prep=0.00s embed=0.10s total=0.10s
[Chronos2Embedding._embed] pipeline=0.74s total=0.84s shape=(77, 256)
[Chronos2Embedding.transform] embed done 0.84s
[Chronos2Embedding._embed] b

In [5]:
svsdf=Sdfsdf3

NameError: name 'Sdfsdf3' is not defined

In [ ]:
from time import perf_counter
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score
from sklearn.linear_model import RidgeClassifierCV
from threadpoolctl import threadpool_limits
from tscglue import data_loader
from tscglue.models_tsfm import Chronos2Embedding, MantisEmbedding

class RidgeClassifierCVDecisionProba(RidgeClassifierCV):
    def fit(self, X, y):
        with threadpool_limits(limits=1):
            return super().fit(X, y)

X_train, y_train, X_test, y_test = data_loader.load_fold(dataset, 12)

transformers = [
    ("MantisV2", MantisEmbedding()),
    ("chronos-bolt-tiny", Chronos2Embedding(model_id="amazon/chronos-bolt-tiny")),
    ("chronos-bolt-mini", Chronos2Embedding(model_id="amazon/chronos-bolt-mini")),
    ("chronos-bolt-small", Chronos2Embedding(model_id="amazon/chronos-bolt-small")),
    ("chronos-bolt-base", Chronos2Embedding(model_id="amazon/chronos-bolt-base")),
    ("chronos-2", Chronos2Embedding(model_id="amazon/chronos-2")),
]

for name, t in transformers:
    t0 = perf_counter()
    Xt_train = t.transform(X_train)
    Xt_test = t.transform(X_test)
    clf = make_pipeline(StandardScaler(), RidgeClassifierCVDecisionProba(alphas=np.logspace(-3, 3, 10)))
    clf.fit(Xt_train, y_train)
    y_pred = clf.predict(Xt_test)
    elapsed = perf_counter() - t0
    print(f"{name}: acc={accuracy_score(y_test, y_pred):.4f} ({elapsed:.1f}s)")